In [8]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from agents.classifying_agent import ClassifyingAgent
import chromadb
from agents.agent import Agent

load_dotenv(override=True)

True

In [9]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collections = client.get_or_create_collection('products')

In [10]:
classifyingAgent = ClassifyingAgent(collection=collections)


# def classifyingAgent(product_description):
#     # agent = Agent(name="classifying_agent", description="Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.", tools=[{"type": "function", "function": classifyingAgent}])
#     # response = agent.run(product_description=product_description)
#     # return response
#     return "baby products, toys, home goods"

categorization_agent = {
    "name": "categorization_agent",
    "description": "Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_description": {
                "type": "string",
                "description": "A description of the product that needs to be categorized."
            },
        },
        "required": ["product_description"],
        "additionalProperties": False
    }
}


ping_manager_function = {
    "name": "ping_manager",
    "description": "Notify a store manager when the customer disagrees with the suggested product category and asks for review.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_description": {
                "type": "string",
                "description": "The product description or product being disputed."
            },
            "reason": {
                "type": "string",
                "description": "Why the customer disagrees with the category, if provided."
            }
        },
        "required": ["product_description"],
        "additionalProperties": False
    }
}

Using device: mps


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:


openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [12]:


tools = [
    {"type": "function", "function": categorization_agent},
    {"type": "function", "function": ping_manager_function},
]

def ping_manager(product_description: str, reason: str = ""):
    print(f"Manager pinged for customer disagreement: {product_description}. Reason: {reason}")
    return "I understand you disagree with the suggested category. I have sent this product to a store manager for review."


def handle_tool_call(message):
    mapping = {
        "categorization_agent": classifyingAgent.classify,
        "ping_manager": ping_manager,
    }

    results = []

    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        tool = mapping.get(tool_name)

        if tool:
            result = tool(**arguments)
        else:
            result = f"Unknown tool: {tool_name}"

        results.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return results


In [13]:
WELCOME_MESSAGE = "Hi! My name is Peeta, and I'm here to help you register your product. First, please provide me with a description of your product. For example, what does it do, what are its features, and any other relevant information. Based on your description, I will sort your product into the correct category."
system_message ="""
You are a helpful assistant for an e-commerce platform that helps businesses register their products. Do not make up anything if you are unsure. Inform the customer when you contacted a store manager for further assistance. If the customer disagrees with the suggested category, call the ping_manager tool and tell them a store manager will review it. You will be provided with a product description, and your task is to categorize the product based on the description. If the product is difficult to categorize, you will return the 3 categories you think it could belong to.
"""

In [ ]:

# from langchain_core import messages


chatbot = gr.Chatbot(
    type="messages",
    value=[
        {"role": "assistant", "content": WELCOME_MESSAGE}
    ]
)

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        # messages.append(message)
        # # messages.append(message.model_dump(exclude_none=True))
        # messages.append(response)
        messages.append(message.model_dump(exclude_none=True))
        messages.extend(response)
        # print("this is messages after appending", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        # print("this is final response", response)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat,chatbot=chatbot ,type="messages").launch()

/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_28066/3872658606.py:4: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DEBUG: ClassifyingAgent LLM called tool: categorize_function
this is the raw response from the Specialist agent Home & Garden
this is the raw response from the RAG agent Home & Garden
this is the category returned from the ensemble Home & Garden
Manager pinged for customer disagreement: a mop pad. Reason: I disagree with the suggested category 'Home & Garden' for a mop pad.
